<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day08-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 8, Segment 3 discussion — does "chance level" change with more classes?

The book page's untrained 5-class network measured accuracy 0.248 (chance for
5 balanced classes is 0.20), MCC 0.086, and macro AUROC 0.542. This notebook
adds a **sixth** real class — Endoplasmic reticulum (UniProt `SL-0095`), not
used anywhere else in this course — and asks: for an *untrained* network,
does adding one more class change what "no better than chance" looks like,
and by how much?

In [1]:
import numpy as np
import requests
import torch
import torch.nn as nn
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
SEQ_LEN = 150

def one_hot_encode(sequence, length=SEQ_LEN):
    encoded = np.zeros((length, len(AMINO_ACIDS)), dtype=np.float32)
    for position, residue in enumerate(sequence[:length]):
        if residue in AA_TO_INDEX:
            encoded[position, AA_TO_INDEX[residue]] = 1.0
    return encoded.flatten()

def fetch_uniprot(sl_code, size=200):
    query = f"organism_id:9606 AND reviewed:true AND cc_scl_term:{sl_code} AND length:[50 TO 500]"
    r = requests.get(
        "https://rest.uniprot.org/uniprotkb/search",
        params={"query": query, "fields": "accession,sequence", "format": "tsv", "size": size},
        timeout=60,
    )
    r.raise_for_status()
    lines = r.text.strip().split("\n")[1:]
    return [line.split("\t")[1] for line in lines if len(line.split("\t")) >= 2 and line.split("\t")[1]]

UNIPROT_CLASSES_6 = {
    "Cytoplasm": "SL-0086",
    "Nucleus": "SL-0191",
    "Mitochondrion": "SL-0173",
    "Secreted": "SL-0243",
    "Cell membrane": "SL-0039",
    "Endoplasmic reticulum": "SL-0095",   # NEW -- not used on the book page
}
CLASS_NAMES_6 = list(UNIPROT_CLASSES_6.keys())

print("Fetching all 6 classes live from UniProt (this takes a moment)...")
sequences_by_class = {name: fetch_uniprot(code) for name, code in UNIPROT_CLASSES_6.items()}
for name, seqs in sequences_by_class.items():
    print(f"  {name}: {len(seqs)} sequences")

Fetching all 6 classes live from UniProt (this takes a moment)...


  Cytoplasm: 200 sequences
  Nucleus: 200 sequences
  Mitochondrion: 200 sequences
  Secreted: 200 sequences
  Cell membrane: 200 sequences
  Endoplasmic reticulum: 200 sequences


In [2]:
N_PER_CLASS = min(len(v) for v in sequences_by_class.values())
print("balancing every class to", N_PER_CLASS, "sequences")

rng = np.random.RandomState(0)
balanced_seqs, balanced_labels = [], []
for class_idx, class_name in enumerate(CLASS_NAMES_6):
    pool = sequences_by_class[class_name]
    chosen = rng.choice(len(pool), N_PER_CLASS, replace=False)
    for i in chosen:
        balanced_seqs.append(pool[i])
        balanced_labels.append(class_idx)

X6 = np.stack([one_hot_encode(s) for s in balanced_seqs])
y6 = np.array(balanced_labels)
print("total balanced 6-class dataset:", len(X6), "sequences,", len(CLASS_NAMES_6), "classes")

X6_train, X6_test, y6_train, y6_test = train_test_split(X6, y6, test_size=0.2, stratify=y6, random_state=0)
print("train:", len(X6_train), " test:", len(X6_test))

balancing every class to 200 sequences
total balanced 6-class dataset: 1200 sequences, 6 classes
train: 960  test: 240


## The same architecture as the book page, untrained, now with 6 outputs

In [3]:
class SubcellularLocalizationClassifier6(nn.Module):
    def __init__(self, input_dim=SEQ_LEN * len(AMINO_ACIDS), hidden_dim=64, n_classes=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x):
        return self.net(x)


model6 = SubcellularLocalizationClassifier6(n_classes=len(CLASS_NAMES_6))

X_test_t = torch.tensor(X6_test, dtype=torch.float32)
with torch.no_grad():
    logits = model6(X_test_t)
    probs = torch.softmax(logits, dim=1).numpy()
preds = probs.argmax(axis=1)

acc6 = (preds == y6_test).mean()
mcc6 = matthews_corrcoef(y6_test, preds)
auroc6 = roc_auc_score(y6_test, probs, multi_class="ovr", average="macro")

print(f"6-class untrained network: accuracy = {acc6:.3f}  (chance = {1/6:.3f})")
print(f"                           MCC      = {mcc6:.3f}  (chance ~= 0)")
print(f"                           macro AUROC = {auroc6:.3f}  (chance = 0.500)")
print()
print("Compare to the book page's 5-class untrained network:")
print("  accuracy = 0.248 (chance = 0.200), MCC = 0.086, macro AUROC = 0.542")

6-class untrained network: accuracy = 0.167  (chance = 0.167)
                           MCC      = 0.000  (chance ~= 0)
                           macro AUROC = 0.529  (chance = 0.500)

Compare to the book page's 5-class untrained network:
  accuracy = 0.248 (chance = 0.200), MCC = 0.086, macro AUROC = 0.542


## What actually changed, read from the real numbers above

Chance-level accuracy drops from $1/5 = 0.200$ to $1/6 \approx 0.167$ — an
untrained network's accuracy should track that, and (per the numbers printed
above) does, within the noise you'd expect from one random initialization on
one dataset. MCC's chance level stays "around zero" regardless of the number
of classes, since it's already normalized for exactly this. Macro AUROC's
chance level stays at 0.5 regardless of class count too, for the same reason
— it's a per-class one-vs-rest measure, averaged, not something that dilutes
as more classes are added.

**Discuss:** if accuracy's chance level depends on the number of classes but
MCC's and AUROC's don't, which of these three metrics would you trust most
to compare two networks trained on datasets with a *different* number of
classes?